# Tái lập các hàng **STSB** của **Table 3** — RAFT (Lukasik et al., ICLR 2025)

*"Better autoregressive regression with LLMs via regression-aware fine-tuning"*, Google Research, ICLR 2025
([proceedings PDF](https://proceedings.iclr.cc/paper_files/paper/2025/file/ae3bb0adcdce28afb5f7e86a79b6cffa-Paper-Conference.pdf), [OpenReview](https://openreview.net/forum?id=xGs7Ch3Vyo)).
Tóm tắt đầy đủ công thức và các chi tiết paper **không** nói rõ: `docs/tom_tat_paper_RAFT.md` trong repo này.

**Mục tiêu (Table 3, p.8, STSB, RMSE; fine-tuned = mean ± std qua 3 lần huấn luyện):**

| Gemma-2 | Zero-shot standard decoding | Zero-shot RAIL | Std FT + standard decoding | Std FT + RAIL | Predictive head | RAFT |
|---|---|---|---|---|---|---|
| 2B | 1.10 | 1.05 | 0.59 ± 0.01 | 0.57 ± 0.00 | 0.54 ± 0.00 | 0.54 ± 0.01 |
| 9B | 1.37 | 1.29 | 0.58 ± 0.00 | 0.53 ± 0.00 | 0.52 ± 0.00 | 0.51 ± 0.01 |

**Sáu cột ↔ code trong notebook này**

| Cột Table 3 | Predictor ŷ(x) | Loss fine-tune | Ở đâu |
|---|---|---|---|
| Zero-shot standard decoding | greedy, ŷ = float(z) (Eq. 1–2) | — | Phần 7 `greedy_predict`, chạy ở Phần 12 |
| Zero-shot RAIL | E_{y~p}[float(y)] (Eq. 4), trung bình 32 mẫu, T = 1 (§5.1) | — | Phần 7 `rail_sampling_predict`, chạy ở Phần 12 |
| Standard FT + standard decoding | greedy (Eq. 1–2) | −log p(str(y*)\|x) (Eq. 7–8) | Phần 13–14, method `ce` |
| Standard FT + RAIL | Eq. 4, 32 mẫu, T = 1 | Eq. 7–8 | Phần 13–14, method `ce` |
| Predictive head | b + f_out(x)_{v*,L} (Table 2) | (ŷ − y*)² | Phần 13–14, method `head` |
| RAFT | Σ_{y∈{1..5}} y·p(str(y)\|x) (Eq. 5, §4.2) | Eq. 10 | Phần 13–14, method `raft` |

**Cách chạy**
* **GPU thật** (Colab A100/H100, hoặc máy riêng): đặt `MODEL_NAME`, có `HF_TOKEN` (Gemma-2 là gated model), *Run all*. Kết quả JSON từng run ở `OUTPUT_DIR/<model>/runs/`, bảng cuối ở `table3_stsb.md`. Chạy bị ngắt thì chạy lại: các run đã xong được bỏ qua (`SKIP_EXISTING`).
* **`SMOKE_TEST`** (tự bật khi không có GPU, hoặc `RAFT_SMOKE_TEST=1`): Gemma-2 *tí hon* khởi tạo ngẫu nhiên + tokenizer mức ký tự + dữ liệu tổng hợp, chạy trên CPU vài phút, **không cần mạng**. Chỉ để kiểm tra code — số liệu không có ý nghĩa so với paper.

> ⚠️ Chi phí: paper huấn luyện 200K bước × 3 LR × 3 seed × 3 kiểu fine-tune × 2 cỡ mô hình. Mặc định ở đây `MAX_STEPS = 6000` (DEVIATION có lý do, xem Phần 14). Notebook **không** tự đảm bảo ra đúng số của paper: nhiều chi tiết paper không công bố (xem Phần 16).

## Phần 0. Cài đặt thư viện
Cần: `torch`, `transformers` (có Gemma-2; ≥ 4.56 để dùng tham số `dtype`, bản cũ hơn vẫn chạy), `datasets`, `peft` (chỉ khi dùng LoRA / unit check), `scipy`. Trên Colab bỏ comment dòng `%pip` bên dưới.

In [ ]:
# %pip install -q "torch>=2.3" "transformers>=4.56" "datasets>=2.20" "accelerate>=0.33" "peft>=0.12" scipy

## Phần 1. Cấu hình
Mọi thiết lập của paper cho STSB/Gemma-2 (App. D p.23, §5.1 p.8, Table 6–7 p.23) nằm trong ô dưới; chỗ nào paper không nói thì ghi lựa chọn của notebook (giải thích ở Phần 16 và trong `docs/`).

* **Mô hình**: §5.1 viết "*Gemma-2 … instruction-tuned model families*" ⇒ mặc định `google/gemma-2-2b-it` / `google/gemma-2-9b-it`; bản base để lựa chọn. Paper không nói có dùng chat template ⇒ cờ `USE_CHAT_TEMPLATE` (mặc định `False`: dùng nguyên prompt Table 6).
* **Tối ưu** (App. D): cập nhật **toàn bộ** tham số, Adafactor (ε1 = 1e-30, ε2 = 1e-3, decay rate = 0.8), LR **hằng** chọn trong {1e-4, 1e-5, 1e-6} theo validation, batch 16, dropout 0.1, 200K bước, chọn bước tốt nhất trên validation.
* **Bộ nhớ**: 2B full fine-tune với trọng số fp32 (master) + autocast bf16 cần ~30 GB ⇒ A100 40/80 GB. 9B fp32 không vừa 1 GPU 80 GB ⇒ dùng `PARAM_DTYPE="bfloat16"` + `STOCHASTIC_ROUNDING=True` (~55–65 GB, nên bật `GRADIENT_CHECKPOINTING`), hoặc `USE_LORA=True` (**sai khác paper**). Lý do cần fp32/stochastic rounding: với LR 1e-5–1e-6 và hệ số tỉ lệ theo tham số của Adafactor, bước cập nhật tương đối ~1e-6 nhỏ hơn nhiều độ phân giải bf16 (~4e-3) nên làm tròn thường sẽ "nuốt" mất cập nhật.

In [ ]:
# ==== CẤU HÌNH (sửa ở đây) ====================================================
# Mỗi tham số ghi kèm nguồn trong paper; "DEVIATION" = cố ý khác paper để chạy được.
import os

# --- Chế độ chạy ---
# None = tự động: SMOKE_TEST khi máy không có GPU CUDA. Biến môi trường RAFT_SMOKE_TEST=1/0 sẽ ghi đè.
SMOKE_TEST = None

# --- Mô hình (§5.1 p.8: "Gemma-2 ... instruction-tuned model families") ---
MODEL_NAME = "google/gemma-2-2b-it"     # hoặc "google/gemma-2-9b-it"; bản base: "google/gemma-2-2b", "google/gemma-2-9b"
HF_TOKEN = os.environ.get("HF_TOKEN")   # Gemma-2 là gated model: cần token đã chấp nhận license
USE_CHAT_TEMPLATE = False               # paper không nói; mặc định dùng prompt Table 6 dạng văn bản thô
OUTPUT_DIR = "raft_stsb_results"

# --- Những gì cần chạy ---
METHODS = ["zero_shot", "ce", "head", "raft"]   # ce = standard fine-tuning (cột 3 và 4)
SEEDS = [0, 1, 2]                               # §5.1: fine-tune 3 lần -> mean ± std
LEARNING_RATES = [1e-4, 1e-5, 1e-6]             # App. D p.23: LR hằng, chọn trên validation

# --- Huấn luyện (App. D p.23) ---
BATCH_SIZE = 16
MICRO_BATCH_SIZE = 16          # < BATCH_SIZE => tích luỹ gradient (tương đương toán học với batch 16)
MAX_STEPS = 6000               # paper: 200_000 bước. DEVIATION — Fig. 4 p.29: RAFT/head hội tụ trong ~1k–4.5k bước
EVAL_EVERY = 200               # Table 21 p.29 / Fig. 5 p.30 dùng bước là bội của 200
EARLY_STOP_PATIENCE = None     # None = tắt (như paper); ví dụ 10 => dừng sau 10 lần eval không cải thiện (DEVIATION)
DROPOUT = 0.1                  # App. D
DROPOUT_MODE = "residual"      # "residual" | "attention" | "both" | "none" — paper không nói dropout đặt ở đâu
ADAFACTOR_KW = dict(           # App. D: eps1=1e-30, eps2=1e-3, decay rate=0.8 (= mặc định T5X Adafactor)
    eps1=1e-30, eps2=1e-3, decay_rate=0.8, clipping_threshold=1.0,
    multiply_by_parameter_scale=True, min_dim_size_to_factor=128)
PARAM_DTYPE = "float32"        # trọng số master. "bfloat16" (+ stochastic rounding) để 9B vừa 1 GPU 80GB
STOCHASTIC_ROUNDING = True     # chỉ có tác dụng khi PARAM_DTYPE="bfloat16"
COMPUTE_DTYPE = "bfloat16"     # autocast khi tính toán
GRADIENT_CHECKPOINTING = False
ATTN_IMPLEMENTATION = "eager"  # khuyến nghị cho Gemma-2 khi train (attention soft-capping)
USE_LORA = False               # DEVIATION nếu True: paper cập nhật toàn bộ tham số (App. D)
LORA_R, LORA_ALPHA = 16, 32

# --- Dữ liệu (Table 6, Table 7 p.23) ---
DATA_SOURCE = "hf"             # "hf" (nyu-mll/glue, stsb) | "csv" (STSB_CSV_DIR) ; SMOKE_TEST dùng dữ liệu tổng hợp
STSB_CSV_DIR = os.environ.get("STSB_CSV_DIR")   # thư mục chứa stsb-en-train.csv, stsb-en-dev.csv (tuỳ chọn)
SPLIT_SEED = 0                 # seed cố định để tách val từ GLUE train (độc lập với seed huấn luyện)
N_VAL = 863                    # Table 7
N_TRAIN = None                 # None = phần còn lại của GLUE train (5,749 - 863 = 4,886; paper ghi 4,887)
TARGET_DECIMALS = 2            # str(y) khi fine-tune CE: làm tròn 2 chữ số thập phân (paper không nói)
ANSWER_PREFIX = None           # None = tự động: " " (prompt thô) hoặc "" (chat template). Xem Phần 5 của notebook

# --- RAFT (§4.1, §5.1 p.8) ---
RAFT_GRID = [1, 2, 3, 4, 5]    # Y_grid mặc định của paper; có thể thử [0, 1, 2, 3, 4, 5]
RAFT_NORMALIZE = False         # Eq. 10 không chuẩn hoá; Table 16 p.28 (PaLM-2) là ablation

# --- Predictive head (Table 2: ŷ = b + f_out(x)_{v*,L}) ---
HEAD_TOKEN = "<unused0>"       # v*: paper không nói token nào; chọn token "unused" của Gemma
HEAD_BIAS_INIT = 0.0

# --- Giải mã / suy luận (§2.2 p.3, §5.1 p.8) ---
RAIL_NUM_SAMPLES = 32          # zero-shot RAIL: 32 mẫu, T = 1
RAIL_TEMPERATURE = 1.0
MAX_NEW_TOKENS = 8
NON_NUMERIC_VALUE = 0.0        # §2.2: float(z) = 0.0 nếu z không phải số trong Y
OUT_OF_RANGE_TO_DEFAULT = True # số nằm ngoài [0,5] cũng coi là "không thuộc Y" -> 0.0
PARSE_MODE = "first_number_first_line"   # hoặc "strict"
CE_EVAL_RAIL_GRID = False      # cột phụ: RAIL bằng chấm điểm lưới 0.0..5.0 (Eq. 5) cho mô hình CE (đắt: 51 chuỗi/ví dụ)
EVAL_BATCH_SIZE = 32
GEN_BATCH_SIZE = 8             # số prompt mỗi lần generate (x RAIL_NUM_SAMPLES chuỗi)
EVAL_SAMPLING_SEED = 1234
STD_DDOF = 1                   # độ lệch chuẩn mẫu qua các seed (paper không nói ddof)
SKIP_EXISTING = True           # bỏ qua run đã có JSON (chạy tiếp sau khi bị ngắt)

## Phần 2. Khởi tạo môi trường
Ghi đè cấu hình cho `SMOKE_TEST`, chọn thiết bị, hàm seed, autocast. Trên máy có GPU mà muốn thử nhanh: đặt `SMOKE_TEST = True` ở Phần 1 (hoặc biến môi trường `RAFT_SMOKE_TEST=1`).

In [ ]:
import copy, csv, gc, glob, json, math, random, re, time
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence

import numpy as np
import torch
import torch.nn.functional as F
import transformers
from packaging import version
from scipy.stats import pearsonr, spearmanr

transformers.utils.logging.disable_progress_bar()   # tránh thanh tiến trình "Loading weights" ở mỗi run

# ---- Chế độ SMOKE_TEST: mô hình Gemma-2 tí hon khởi tạo ngẫu nhiên + dữ liệu tổng hợp, chạy CPU, không cần mạng ----
_env = os.environ.get("RAFT_SMOKE_TEST")
if _env is not None:
    SMOKE_TEST = _env.strip() == "1"
if SMOKE_TEST is None:
    SMOKE_TEST = not torch.cuda.is_available()
OUTPUT_DIR = os.environ.get("RAFT_OUTPUT_DIR", OUTPUT_DIR)

if SMOKE_TEST:
    OUTPUT_DIR = os.path.join(OUTPUT_DIR, "smoke")
    MODEL_NAME = os.path.join(OUTPUT_DIR, "_assets", "tiny-gemma2")
    DATA_SOURCE = "synthetic"
    SEEDS = [0, 1]
    LEARNING_RATES = [3e-2, 3e-3]
    BATCH_SIZE, MICRO_BATCH_SIZE = 16, 8
    MAX_STEPS, EVAL_EVERY = 30, 10
    N_VAL, N_TRAIN = 24, None
    PARAM_DTYPE, COMPUTE_DTYPE = "float32", "float32"
    GRADIENT_CHECKPOINTING, USE_LORA = False, False
    CE_EVAL_RAIL_GRID = True       # chạy thử cả cột phụ
    EVAL_BATCH_SIZE, GEN_BATCH_SIZE = 16, 8
    RAIL_NUM_SAMPLES = 8           # cho nhanh trên CPU (cấu hình thật: 32)
    HEAD_TOKEN = "<unused0>"
else:
    OUTPUT_DIR = os.path.join(OUTPUT_DIR, MODEL_NAME.rstrip("/").split("/")[-1])   # mỗi mô hình một thư mục
    if HF_TOKEN is None:
        try:   # Colab: Secrets -> HF_TOKEN
            from google.colab import userdata
            HF_TOKEN = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if HF_TOKEN is None:
        print("Lưu ý: chưa thấy HF_TOKEN. Gemma-2 là gated model: chấp nhận license trên Hugging Face rồi "
              "đặt biến môi trường HF_TOKEN hoặc chạy `huggingface-cli login`.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PARAM_TORCH_DTYPE = getattr(torch, PARAM_DTYPE)
COMPUTE_TORCH_DTYPE = getattr(torch, COMPUTE_DTYPE)
RUNS_DIR = os.path.join(OUTPUT_DIR, "runs")
os.makedirs(RUNS_DIR, exist_ok=True)
Y_MIN, Y_MAX = 0.0, 5.0        # STSB: điểm trong [0, 5] (Table 6)
if ANSWER_PREFIX is None:
    ANSWER_PREFIX = "" if USE_CHAT_TEMPLATE else " "


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def autocast_ctx():
    """bf16 autocast cho phần tính toán; trọng số master giữ theo PARAM_DTYPE."""
    return torch.autocast(device_type=DEVICE.type, dtype=COMPUTE_TORCH_DTYPE,
                          enabled=COMPUTE_TORCH_DTYPE != torch.float32)


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print(f"SMOKE_TEST={SMOKE_TEST} | device={DEVICE} | torch {torch.__version__} | transformers {transformers.__version__}")
print(f"MODEL_NAME={MODEL_NAME} | OUTPUT_DIR={OUTPUT_DIR} | params {PARAM_DTYPE}, compute {COMPUTE_DTYPE}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB")

## Phần 3. Tài nguyên cho SMOKE_TEST và unit check
* `build_char_tokenizer()`: tokenizer mức ký tự — mỗi chữ số và dấu cách là một token, giống tính chất của tokenizer Gemma-2 (SentencePiece 256k, *split digits*, *preserved whitespace* — Gemma 2 tech report; với tokenizer thật `" 3"` → `['▁', '3']`).
* `build_tiny_gemma2()`: `Gemma2ForCausalLM` 2 lớp, hidden 64 — cùng code đường đi với Gemma-2 thật (final-logit soft-capping 30, attention soft-capping 50, sliding window, embedding buộc E_in = E_out).
* `make_synthetic_stsb()`: cặp câu giả, nhãn ∈ [0, 5].

In [ ]:
from tokenizers import Regex, Tokenizer, decoders, pre_tokenizers, processors
from tokenizers import models as tk_models
from transformers import Gemma2Config, Gemma2ForCausalLM, PreTrainedTokenizerFast


def build_char_tokenizer():
    """Tokenizer mức ký tự: mỗi chữ số và dấu cách là 1 token (giống Gemma tách chữ số, khoảng trắng là token riêng)."""
    specials = ["<pad>", "<eos>", "<bos>", "<unk>", "<unused0>"]
    vocab = {t: i for i, t in enumerate(specials)}
    for c in [chr(i) for i in range(32, 127)] + ["\n"]:
        vocab.setdefault(c, len(vocab))
    tk = Tokenizer(tk_models.WordLevel(vocab, unk_token="<unk>"))
    tk.pre_tokenizer = pre_tokenizers.Split(Regex("."), behavior="isolated")
    tk.post_processor = processors.TemplateProcessing(single="<bos> $A", special_tokens=[("<bos>", vocab["<bos>"])])
    tk.decoder = decoders.Fuse()
    return PreTrainedTokenizerFast(tokenizer_object=tk, bos_token="<bos>", eos_token="<eos>",
                                   pad_token="<pad>", unk_token="<unk>")


def build_tiny_gemma2(tok, seed=0):
    """Gemma-2 tí hon (cùng kiến trúc: soft-capping, sliding window, norm sau mỗi sub-layer), khởi tạo ngẫu nhiên."""
    cfg = Gemma2Config(vocab_size=len(tok), hidden_size=64, intermediate_size=128, num_hidden_layers=2,
                       num_attention_heads=4, num_key_value_heads=2, head_dim=16, max_position_embeddings=1024,
                       sliding_window=128, query_pre_attn_scalar=16, pad_token_id=tok.pad_token_id,
                       eos_token_id=tok.eos_token_id, bos_token_id=tok.bos_token_id, tie_word_embeddings=True)
    torch.manual_seed(seed)
    return Gemma2ForCausalLM(cfg)


def make_synthetic_stsb(n, seed):
    """Cặp câu giả; nhãn = 5 * Jaccard(từ), làm tròn 2 chữ số — chỉ để kiểm tra pipeline."""
    rng = random.Random(seed)
    words = "a man woman dog cat is playing eating running guitar flute the red big small ball".split()
    out = []
    for i in range(n):
        s1 = rng.sample(words, 4)
        k = rng.randint(0, 4)
        s2 = s1[:k] + rng.sample(words, 4 - k)
        jac = len(set(s1) & set(s2)) / len(set(s1) | set(s2))
        out.append(dict(sentence1=" ".join(s1) + ".", sentence2=" ".join(s2) + ".", label=round(5 * jac, 2), idx=i))
    return out


if SMOKE_TEST and not os.path.exists(os.path.join(MODEL_NAME, "config.json")):
    _tok = build_char_tokenizer()
    build_tiny_gemma2(_tok, seed=0).save_pretrained(MODEL_NAME)
    _tok.save_pretrained(MODEL_NAME)
    print("Đã tạo mô hình tí hon tại", MODEL_NAME)

## Phần 4. Dữ liệu STSB (Table 6, Table 7 p.23)
* **Prompt** — nguyên văn Table 6: *"What is the sentence similarity between the following two sentences measured on a scale of 0 to 5: {Sentence #1}, {Sentence #2}. The similarity measured on a scale of 0 to 5 with 0 being unrelated and 5 being related is equal to"*. Khoảng giá trị đích [0, 5].
* **Chia tập** — Table 7: train 4,887 / validation 863 / test 1,500. GLUE STS-B có train 5,749, validation 1,500, test 1,379 (nhãn test bị ẩn) ⇒ *test của paper = GLUE validation*, train/val tách từ GLUE train. Lưu ý 4,887 + 863 = 5,750 ≠ 5,749: lệch 1, paper không giải thích. Notebook giữ đúng val = 863 (dùng để chọn mô hình), train = 4,886 phần còn lại; cách tách: hoán vị bằng `random.Random(SPLIT_SEED=0)` (paper không nói cách tách). Chỉ số GLUE `idx` của từng tập được lưu ở `split_indices.json`.
* **str(y) cho fine-tuning CE** (Eq. 8): nhãn GLUE lưu dạng float32 (3.8 → 3.7999999…); ~9% nhãn có 2–3 chữ số thập phân (vd 4.25, 2.333) và vài nhãn là số thập phân vô hạn tuần hoàn. Paper không nói độ chính xác ⇒ `format_target`: làm tròn `TARGET_DECIMALS = 2` chữ số, dạng ngắn nhất (`'5.0'`, `'3.8'`, `'4.25'`, `'2.33'`), sai số làm tròn RMSE ≈ 5e-4.
* `DATA_SOURCE = "csv"` cho phép đọc bản CSV của STS benchmark (vd [PhilipMay/stsb-multi-mt](https://github.com/PhilipMay/stsb-multi-mt), cùng 5,749/1,500 cặp) khi không truy cập được Hugging Face; thứ tự dòng có thể khác GLUE nên tập train/val tách ra có thể khác.

In [ ]:
# Prompt STSB — nguyên văn Table 6 (p.23). {s1}, {s2} = Sentence #1, Sentence #2.
PROMPT_TEMPLATE = ("What is the sentence similarity between the following two sentences measured on a scale of 0 to 5: "
                   "{s1}, {s2}. The similarity measured on a scale of 0 to 5 with 0 being unrelated and 5 being "
                   "related is equal to")


@dataclass
class Example:
    s1: str
    s2: str
    label: float
    idx: int = -1

    def prompt(self) -> str:
        return PROMPT_TEMPLATE.format(s1=self.s1, s2=self.s2)


def format_target(y: float) -> str:
    """str(y) (§2.1) cho fine-tuning CE: làm tròn TARGET_DECIMALS chữ số, dạng ngắn nhất ('5.0', '3.8', '4.25')."""
    return repr(round(float(y), TARGET_DECIMALS))


def _rows_to_examples(rows):
    return [Example(r["sentence1"].strip(), r["sentence2"].strip(), float(r["label"]), int(r.get("idx", i)))
            for i, r in enumerate(rows)]


def load_glue_stsb():
    """GLUE STS-B: train (5,749) -> tách train/val; validation (1,500) = test (nhãn test GLUE bị ẩn)."""
    if DATA_SOURCE == "synthetic":
        return _rows_to_examples(make_synthetic_stsb(64 + N_VAL, seed=1)), _rows_to_examples(make_synthetic_stsb(24, seed=2))
    if DATA_SOURCE == "csv":   # ví dụ github.com/PhilipMay/stsb-multi-mt (cột: sentence1, sentence2, score)
        def read(name):
            with open(os.path.join(STSB_CSV_DIR, name), encoding="utf-8") as f:
                return [dict(sentence1=r[0], sentence2=r[1], label=float(r[2]), idx=i)
                        for i, r in enumerate(csv.reader(f)) if r]
        return _rows_to_examples(read("stsb-en-train.csv")), _rows_to_examples(read("stsb-en-dev.csv"))
    from datasets import load_dataset
    ds = load_dataset("nyu-mll/glue", "stsb")
    return _rows_to_examples(ds["train"]), _rows_to_examples(ds["validation"])


def split_train_val(pool, n_val, n_train, seed):
    """Tách tất định: hoán vị pool bằng random.Random(seed); n_val phần tử đầu = val, phần còn lại = train."""
    idx = list(range(len(pool)))
    random.Random(seed).shuffle(idx)
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    if n_train is not None:
        train_idx = train_idx[:n_train]
    return [pool[i] for i in train_idx], [pool[i] for i in val_idx]


glue_train, TEST = load_glue_stsb()
TRAIN, VAL = split_train_val(glue_train, N_VAL, N_TRAIN, SPLIT_SEED)
if DATA_SOURCE != "synthetic":
    if len(glue_train) != 5749 or len(TEST) != 1500:
        print(f"CẢNH BÁO: kích thước GLUE khác mong đợi: train={len(glue_train)}, validation={len(TEST)}")
with open(os.path.join(OUTPUT_DIR, "split_indices.json"), "w") as f:
    json.dump({"split_seed": SPLIT_SEED, "train_idx": [e.idx for e in TRAIN], "val_idx": [e.idx for e in VAL],
               "test_idx": [e.idx for e in TEST]}, f)


def label_stats(name, exs):
    y = np.array([e.label for e in exs])
    multi = np.mean([round(v, 1) != round(v, 3) for v in y])
    print(f"{name:5s} n={len(y):5d}  mean={y.mean():.3f}  std={y.std():.3f}  P(y<1)={np.mean(y < 1):.3f}  "
          f"P(>1 chữ số thập phân)={multi:.3f}")


for _n, _e in (("train", TRAIN), ("val", VAL), ("test", TEST)):
    label_stats(_n, _e)
print("\nVí dụ input x (prompt Table 6 + ANSWER_PREFIX):", repr(TRAIN[0].prompt() + ANSWER_PREFIX))
print("Đích CE str(y*):", repr(format_target(TRAIN[0].label)), "(nhãn gốc", TRAIN[0].label, ")")

## Phần 5. Tokenizer, input x và mô hình
**Input x và vị trí L.** Với Gemma-2, dấu cách trước số là token riêng: `"equal to 3.8"` → `[..., '▁to', '▁', '3', '.', '8']`. Nếu x kết thúc ở `"...is equal to"` thì token kế tiếp là `'▁'`, không phải chữ số; khi đó p(str(y)|x) của RAFT một-chữ-số (§4.2: `p_out(·|x)_{str(y),L}`) gần như bằng 0 và ŷ sụp về 0. Fig. 7 (p.31) cho thấy trước fine-tune gần như toàn bộ khối xác suất nằm trên các chữ số ⇒ notebook đặt **x = prompt + `ANSWER_PREFIX`** với `ANSWER_PREFIX = " "` (hoặc `""` nếu dùng chat template), để chữ số là token kế tiếp ngay tại L. Cùng một x cho mọi phương pháp. Phần 12 in ra khối xác suất trên chữ số ở bước 0 để kiểm tra.

**Đích CE.** `str(y*)` + `<eos>` (cần `<eos>` để greedy decoding dừng và để p(str(y)|x) là xác suất của *cả chuỗi*).

**Gemma-2 trên HF.** `attn_implementation="eager"` (Gemma team khuyến nghị fine-tune bằng eager vì attention soft-capping); logit cuối có soft-capping `30·tanh(z/30)` — ta áp lại đúng phép này khi tự tính logits.

**Dropout 0.1.** HF Gemma-2 không có dropout residual, chỉ có `attention_dropout` (mặc định 0). Paper không nói dropout đặt ở đâu ⇒ mặc định `DROPOUT_MODE="residual"`: dropout trên đầu ra mỗi nhánh attention/MLP (sau post-norm của Gemma-2) ngay trước khi cộng vào residual stream, như "residual dropout" của Transformer gốc; có thể chọn `"attention"`/`"both"`/`"none"`.

**Predictive head** (Table 2, *Final-token logit*, Fernandes et al. 2023): ŷ(x) = b + f_out(x)_{v*,L}. Table 4 (p.9) "Special-token logit" Wireless 0.51 (2B) / 0.46 (9B) trùng cột *Predictive head* của Table 3 ⇒ đây đúng là predictor của cột này. Paper không nói v* là token nào ⇒ `<unused0>` (token "unused" của Gemma, tương tự `<extra_id_*>` của T5 trong RankT5). b khởi tạo 0 (Fig. 5 p.30: predictive head bắt đầu ở RMSE ≈ 4.1 trên Wireless, khớp với dự đoán ban đầu ≈ 0).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


def _dtype_kw(dtype):
    # transformers >= 4.56 dùng `dtype`, bản cũ dùng `torch_dtype`
    new = version.parse(transformers.__version__) >= version.parse("4.56")
    return {"dtype": dtype} if new else {"torch_dtype": dtype}


def load_tokenizer(name=None):
    tok = AutoTokenizer.from_pretrained(name or MODEL_NAME, token=HF_TOKEN)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    return tok


def encode_prompt(tok, ex: Example) -> List[int]:
    """Token của input x = prompt Table 6 (+ ANSWER_PREFIX). Với Gemma, ANSWER_PREFIX=' ' là token '▁' riêng,
    nhờ đó chữ số là token kế tiếp ngay tại vị trí L (cần cho RAFT một-chữ-số, §4.2)."""
    if USE_CHAT_TEMPLATE:
        text = tok.apply_chat_template([{"role": "user", "content": ex.prompt()}], tokenize=False,
                                       add_generation_prompt=True) + ANSWER_PREFIX
        return tok(text, add_special_tokens=False)["input_ids"]    # template đã có <bos>
    return tok(ex.prompt() + ANSWER_PREFIX)["input_ids"]          # tokenizer tự thêm <bos>


def encode_target(tok, y: float) -> List[int]:
    """Token của str(y*) + <eos> cho loss CE (Eq. 8)."""
    return tok(format_target(y), add_special_tokens=False)["input_ids"] + [tok.eos_token_id]


def unwrap(model):
    """Bỏ lớp PEFT (LoRA được chèn tại chỗ nên gọi module gốc vẫn dùng adapter)."""
    return model.get_base_model() if hasattr(model, "get_base_model") else model


def _dropout_hook(p):
    def hook(module, inputs, output):
        if not module.training or p <= 0:
            return output
        if isinstance(output, tuple):
            return (F.dropout(output[0], p, True),) + tuple(output[1:])
        return F.dropout(output, p, True)
    return hook


def add_dropout(model, p, mode):
    """Gemma-2 trên HF không có dropout residual; chỉ có `attention_dropout` (mặc định 0).
    'residual': dropout lên đầu ra mỗi nhánh attention/MLP ngay trước khi cộng vào residual (Vaswani et al. 2017)."""
    core = unwrap(model)
    if mode in ("attention", "both"):
        core.config.attention_dropout = p
        for m in core.modules():
            if hasattr(m, "attention_dropout"):
                m.attention_dropout = p
    handles = []
    if mode in ("residual", "both"):
        for layer in core.get_decoder().layers:
            if hasattr(layer, "post_feedforward_layernorm"):   # Gemma-2: x + post_norm(sublayer(pre_norm(x)))
                targets = [layer.post_attention_layernorm, layer.post_feedforward_layernorm]
            else:                                               # kiến trúc kiểu Llama
                targets = [layer.self_attn, layer.mlp]
            handles += [t.register_forward_hook(_dropout_hook(p)) for t in targets]
    return handles


def load_model(name=None):
    model = AutoModelForCausalLM.from_pretrained(name or MODEL_NAME, token=HF_TOKEN,
                                                 attn_implementation=ATTN_IMPLEMENTATION, **_dtype_kw(PARAM_TORCH_DTYPE))
    model.to(DEVICE)
    model.config.use_cache = False
    add_dropout(model, DROPOUT, DROPOUT_MODE)
    if GRADIENT_CHECKPOINTING:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    if USE_LORA:   # DEVIATION so với paper (paper cập nhật toàn bộ tham số)
        from peft import LoraConfig, get_peft_model
        if GRADIENT_CHECKPOINTING:
            model.enable_input_require_grads()
        model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.0,
                                                 target_modules="all-linear", task_type="CAUSAL_LM"))
    return model


class SpecialTokenHead(torch.nn.Module):
    """Predictive head của Fernandes et al. (2023) theo Table 2: ŷ(x) = b + f_out(x)_{v*, L}.
    Tham số học được: b (ở đây) + toàn bộ LLM (gồm hàng embedding của v*, vì Gemma buộc E_in = E_out)."""

    def __init__(self, token_id: int, bias_init: float = 0.0):
        super().__init__()
        self.token_id = int(token_id)
        self.bias = torch.nn.Parameter(torch.tensor(float(bias_init)))

## Phần 6. Tính xác suất mà không tạo logits [B, T, 256k]
Ký hiệu §2.5 (p.4–5): ε_out(x) ∈ ℝ^{D×L} là embedding đầu ra; $f_{\mathrm{out}}(x) = E_{\mathrm{out}}^\top \epsilon_{\mathrm{out}}(x) \in \mathbb{R}^{V\times L}$; $p_{\mathrm{out}}(\cdot\mid x) = \mathrm{softmax}(f_{\mathrm{out}}(x))$.
Với V = 256,000, logits đầy đủ cho batch 16 × ~78 token (x trung bình 73.9 token với tokenizer Gemma-2 + ~4 token đích) là ~320M số (~1.3 GB fp32, chưa kể log-softmax và gradient). Ta chạy decoder một lần lấy hidden state rồi **chỉ nhân E_out tại vài vị trí cần**:
* CE (Eq. 8): các vị trí dự đoán token đích (≤ ~6 vị trí/ví dụ).
* RAFT một-chữ-số: đúng 1 vị trí L (softmax đầy đủ trên V tại L — cần cho mẫu số của softmax).
* Predictive head: chỉ 1 logit f_out(x)_{v*,L} = ⟨ε_out(x)_L, E[v*]⟩ (rồi soft-capping).
* Lưới nhiều token (cột phụ, 0.0..5.0): trie tiền tố chung để chỉ chạy tập chuỗi "phủ" tối thiểu.

In [ ]:
def right_pad(seqs, pad_id):
    T = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), T), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), T), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s, dtype=torch.long)
        mask[i, :len(s)] = 1
    return ids.to(DEVICE), mask.to(DEVICE)


def left_pad(seqs, pad_id):
    T = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), T), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), T), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, T - len(s):] = torch.tensor(s, dtype=torch.long)
        mask[i, T - len(s):] = 1
    return ids.to(DEVICE), mask.to(DEVICE)


def decoder_hidden(model, ids, mask):
    """ε_out(x): hidden state cuối (sau final norm), [B, T, D]. KHÔNG nhân với E_out ở mọi vị trí."""
    return unwrap(model).get_decoder()(input_ids=ids, attention_mask=mask, use_cache=False).last_hidden_state


def _softcap(logits, core):
    cap = getattr(core.config, "final_logit_softcapping", None)   # Gemma-2: 30.0
    return torch.tanh(logits / cap) * cap if cap else logits


def _head_linear(head, h, rows=None):
    """E_out^T h ở độ chính xác của trọng số master (fp32 mặc định), KHÔNG để autocast hạ xuống bf16:
    chỉ vài vị trí nên rẻ, và tránh sai số ~0.4% của bf16 trên logits / xác suất chữ số / ŷ."""
    w = head.weight if rows is None else head.weight[rows]
    b = getattr(head, "bias", None)
    b = None if b is None else (b if rows is None else b[rows])
    with torch.autocast(device_type=DEVICE.type, enabled=False):
        return F.linear(h.to(w.dtype), w, b).float()


def lm_logits(model, h):
    """f_out tại các vị trí đã chọn: E_out^T h (+ soft-capping của Gemma-2), float32. h: [..., D] -> [..., V]."""
    core = unwrap(model)
    return _softcap(_head_linear(core.get_output_embeddings(), h), core)


def token_logit(model, h, token_id):
    """Chỉ một logit f_out(x)_{v,·} — không cần cả vocab 256k."""
    core = unwrap(model)
    rows = slice(token_id, token_id + 1)
    return _softcap(_head_linear(core.get_output_embeddings(), h, rows), core).squeeze(-1)


def last_position_hidden(model, prompts, pad_id):
    """Hidden state tại vị trí cuối L của mỗi input x (right padding)."""
    ids, mask = right_pad(prompts, pad_id)
    h = decoder_hidden(model, ids, mask)
    L = torch.tensor([len(p) - 1 for p in prompts], device=DEVICE)
    return h[torch.arange(len(prompts), device=DEVICE), L]


def target_logprobs(model, prompts, targets, pad_id):
    """Teacher forcing: log p(t_j | x, t_<j) cho từng token đích. Trả về (logp [B,J], mask [B,J])."""
    ids, attn = right_pad([p + t for p, t in zip(prompts, targets)], pad_id)
    h = decoder_hidden(model, ids, attn)
    B, J = len(prompts), max(len(t) for t in targets)
    plen = torch.tensor([len(p) for p in prompts], device=DEVICE)
    rows = torch.arange(B, device=DEVICE)[:, None].expand(B, J)
    cols = (plen[:, None] - 1 + torch.arange(J, device=DEVICE)[None, :]).clamp(max=ids.shape[1] - 1)
    tok_ids = torch.zeros(B, J, dtype=torch.long, device=DEVICE)
    mask = torch.zeros(B, J, device=DEVICE)
    for i, t in enumerate(targets):
        tok_ids[i, :len(t)] = torch.tensor(t, device=DEVICE)
        mask[i, :len(t)] = 1
    logp = F.log_softmax(lm_logits(model, h[rows, cols]), dim=-1)          # chỉ B*J vị trí
    return logp.gather(-1, tok_ids[..., None]).squeeze(-1) * mask, mask


class GridScorer:
    """log p(str(y) | x) cho mọi y trong lưới Y_grid (Eq. 5 / Eq. 10).

    Lưới một-chữ-số (mặc định {1..5}): mỗi str(y) là 1 token => chỉ 1 forward, đọc softmax tại vị trí L:
    p(str(y)|x) = p_out(.|x)_{str(y),L} (§4.2). Lưới nhiều token (vd 0.0..5.0): dùng trie các tiền tố chung,
    chỉ chạy tập chuỗi "phủ" tối thiểu thay vì một forward cho mỗi ứng viên."""

    def __init__(self, tok, strings: Sequence[str], values: Sequence[float], append_eos: bool = False):
        self.strings = list(strings)
        self.values = torch.tensor([float(v) for v in values], dtype=torch.float32)
        cands = []
        for s in strings:
            ids = tok(s, add_special_tokens=False)["input_ids"]
            cands.append(tuple(ids + ([tok.eos_token_id] if append_eos else [])))
        if len(set(cands)) != len(cands):
            raise ValueError("Các giá trị trong lưới có tokenization trùng nhau")
        self.candidates = cands
        self.single_token = all(len(c) == 1 for c in cands)
        needed = {c[:k] for c in cands for k in range(len(c))}               # tiền tố cần phân phối token kế tiếp
        self.covers = sorted(p for p in needed if not any(len(q) > len(p) and q[:len(p)] == p for q in needed))
        loc, pos_cover, pos_off = {}, [], []
        for ci, cov in enumerate(self.covers):
            for o in range(len(cov) + 1):
                pre = cov[:o]
                if pre in needed and pre not in loc:
                    loc[pre] = len(pos_cover)
                    pos_cover.append(ci)
                    pos_off.append(o)
        J = max(len(c) for c in cands)
        self.pos_cover, self.pos_off = torch.tensor(pos_cover), torch.tensor(pos_off)
        self.cand_pos = torch.zeros(len(cands), J, dtype=torch.long)
        self.cand_tok = torch.zeros(len(cands), J, dtype=torch.long)
        self.cand_mask = torch.zeros(len(cands), J)
        for i, c in enumerate(cands):
            for j, t in enumerate(c):
                self.cand_pos[i, j], self.cand_tok[i, j], self.cand_mask[i, j] = loc[c[:j]], t, 1.0

    def log_probs(self, model, prompts, pad_id):
        """[B, |Y_grid|] = log p(str(y) | x)."""
        K = len(self.covers)
        ids, attn = right_pad([p + list(cov) for p in prompts for cov in self.covers], pad_id)
        h = decoder_hidden(model, ids, attn)
        B = len(prompts)
        plen = torch.tensor([len(p) for p in prompts], device=DEVICE)
        rows = torch.arange(B, device=DEVICE)[:, None] * K + self.pos_cover.to(DEVICE)[None, :]
        cols = plen[:, None] - 1 + self.pos_off.to(DEVICE)[None, :]
        logp = F.log_softmax(lm_logits(model, h[rows, cols]), dim=-1)          # [B, U, V], U = số vị trí cần
        lp = logp[:, self.cand_pos.to(DEVICE), self.cand_tok.to(DEVICE)]      # [B, C, J]
        return (lp * self.cand_mask.to(DEVICE)).sum(-1)

## Phần 7. Predictor và hàm mất mát (liên kết với công thức của paper)
* **Eq. 1 (p.3)** $\hat{y}_{\mathrm{AR}}(x) \doteq \mathtt{float}(z),\; z \sim p(\cdot\mid x)$; "*Unless otherwise stated, we assume* float(z) = 0.0 *for* z ∉ 𝒴" (§2.2). Notebook: lấy số đầu tiên trên dòng đầu của phần sinh; không có số hoặc số ∉ [0, 5] ⇒ 0.0, và **ghi lại tỉ lệ này** (`unparsed_rate`) vì rất nhạy: trên test (E[y²] ≈ 7.8), nếu 1% dự đoán bị thay bằng 0.0 thì MSE tăng ≈ 0.01·7.8 ≈ 0.08, tức RMSE 0.55 → ≈ 0.62.
* **Eq. 2** $\hat{y}_{\mathrm{mode}}(x) := \arg\max_{y\in\mathcal{Y}} p(y\mid x)$ — greedy decoding là xấp xỉ (zero-shot standard decoding dùng greedy, §5.1).
* **Eq. 4** $\hat{y}_{\mathrm{RAIL}}(x) = \mathbb{E}_{y\sim p(\cdot\mid x)}[\mathtt{float}(y)]$ — §5.1: "*in zero-shot RAIL, the predictor is obtained by sampling 32 targets with temperature T = 1*" ⇒ trung bình float của 32 mẫu (sampling thuần, top-k = 0, top-p = 1).
* **Eq. 5** $\hat{y}_{\mathrm{RAIL}}(x;\mathcal{Y}_{\mathrm{grid}}) = \sum_{y\in\mathcal{Y}_{\mathrm{grid}}} p(\mathtt{str}(y)\mid x)\cdot y$ — với lưới một-chữ-số $= \sum_y y\cdot p_{\mathrm{out}}(\cdot\mid x)_{\mathtt{str}(y),L}$ (§4.2 p.6). Đây là predictor của RAFT.
* **Eq. 7–8** $\hat{L}(p) = \frac{1}{N}\sum_{(x,y^*)\in S}\ell(y^*, p(\cdot\mid x))$, $\ell(y^*, p(\cdot\mid x)) = -\log p(\mathtt{str}(y^*)\mid x)$ — tổng log-prob các token của str(y*) (+ `<eos>`), trung bình trên ví dụ.
* **Eq. 10** $\hat{L}_{\mathrm{RAFT}}(p;\mathcal{Y}_{\mathrm{grid}}) = \frac{1}{N}\sum_{(x,y^*)\in S}\Big(y^* - \sum_{y\in\mathcal{Y}_{\mathrm{grid}}} p(\mathtt{str}(y)\mid x)\cdot y\Big)^2$, với $\mathcal{Y}_{\mathrm{grid}} = \{1,2,3,4,5\}$ (§5.1 p.8). **Không** chuẩn hoá xác suất trên lưới (đúng Eq. 10; chuẩn hoá chỉ là ablation Table 16 trên PaLM-2 và với lưới {1..5} sẽ ép ŷ ≥ 1 trong khi ~23% nhãn test < 1). Ghi chú: vì số hạng y = 0 đóng góp 0·p('0') = 0, lưới {0,…,5} và {1,…,5} cho **cùng** predictor/loss khi không chuẩn hoá.
* **Predictive head** (Table 2 p.7): $\hat{y}(x) = b + f_{\mathrm{out}}(x)_{v^*,L}$, loss $(\hat{y}(x) - y^*)^2$.

In [ ]:
from transformers import GenerationConfig

_NUM_RE = re.compile(r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)")


def float_of(z: str):
    """float(z) của §2.2: chuỗi -> số; nếu z không biểu diễn một phần tử của Y=[0,5] thì trả 0.0.
    Trả về (giá_trị, parsed_ok)."""
    if PARSE_MODE == "strict":
        m = _NUM_RE.fullmatch(z.strip())
    else:   # "first_number_first_line": số đầu tiên trên dòng đầu của phần sinh ra
        m = _NUM_RE.search(z.split("\n")[0])
    if m is None:
        return NON_NUMERIC_VALUE, False
    v = float(m.group())
    if OUT_OF_RANGE_TO_DEFAULT and not (Y_MIN <= v <= Y_MAX):
        return NON_NUMERIC_VALUE, False
    return v, True


def _eos_ids(model, tok):
    ids = {tok.eos_token_id}
    gen_eos = getattr(unwrap(model).generation_config, "eos_token_id", None)
    if gen_eos is not None:
        ids |= set(gen_eos if isinstance(gen_eos, (list, tuple)) else [gen_eos])
    return sorted(i for i in ids if i is not None)


@torch.no_grad()
def generate_texts(model, tok, prompts, do_sample, num_samples=1):
    """z ~ p(.|x) sinh tự hồi quy (Eq. 1). Greedy (do_sample=False) xấp xỉ mode (Eq. 2);
    sampling thuần ở nhiệt độ T (top_k=0, top_p=1). Trả về list (theo prompt) các list chuỗi."""
    gc_kw = dict(max_new_tokens=MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id, eos_token_id=_eos_ids(model, tok),
                 use_cache=True)
    if do_sample:
        gc_kw.update(do_sample=True, temperature=RAIL_TEMPERATURE, top_k=0, top_p=1.0, num_return_sequences=num_samples)
    else:
        gc_kw.update(do_sample=False, num_return_sequences=1)
    out_texts = []
    for i in range(0, len(prompts), GEN_BATCH_SIZE):
        chunk = prompts[i:i + GEN_BATCH_SIZE]
        ids, attn = left_pad(chunk, tok.pad_token_id)
        with autocast_ctx():
            out = model.generate(input_ids=ids, attention_mask=attn, generation_config=GenerationConfig(**gc_kw))
        texts = tok.batch_decode(out[:, ids.shape[1]:], skip_special_tokens=True)
        k = num_samples if do_sample else 1
        out_texts += [texts[j * k:(j + 1) * k] for j in range(len(chunk))]
    return out_texts


def greedy_predict(model, tok, prompts):
    """Cột 'standard decoding': ŷ_AR(x) = float(z), z = greedy (Eq. 1–2)."""
    vals = [float_of(t[0]) for t in generate_texts(model, tok, prompts, do_sample=False)]
    return [v for v, _ in vals], float(np.mean([not ok for _, ok in vals]))


def rail_sampling_predict(model, tok, prompts):
    """Cột 'RAIL': ŷ_RAIL(x) = E_{y~p(.|x)}[float(y)] (Eq. 4), ước lượng bằng trung bình RAIL_NUM_SAMPLES mẫu ở T=1."""
    preds, bad = [], []
    for samples in generate_texts(model, tok, prompts, do_sample=True, num_samples=RAIL_NUM_SAMPLES):
        vals = [float_of(s) for s in samples]
        preds.append(float(np.mean([v for v, _ in vals])))
        bad += [not ok for _, ok in vals]
    return preds, float(np.mean(bad))


def grid_predict(model, scorer: GridScorer, prompts, pad_id, normalize=False):
    """ŷ_RAIL(x; Y_grid) = Σ_{y∈Y_grid} p(str(y)|x)·y (Eq. 5) — cũng là predictor của RAFT (Eq. 10).
    normalize=True: chia cho Σ_y p(str(y)|x) (ablation Table 16)."""
    logp = scorer.log_probs(model, prompts, pad_id)
    p = torch.softmax(logp, -1) if normalize else logp.exp()
    return (p * scorer.values.to(p.device)).sum(-1), logp      # nhân-cộng fp32 (matmul sẽ bị autocast xuống bf16)


def head_predict(model, head: SpecialTokenHead, prompts, pad_id):
    """Predictive head (Table 2, 'Final-token logit'): ŷ(x) = b + f_out(x)_{v*, L}."""
    h = last_position_hidden(model, prompts, pad_id)
    return head.bias + token_logit(model, h, head.token_id)


# ---- Hàm mất mát theo từng ví dụ (tổng trên micro-batch rồi chia BATCH_SIZE => trung bình 1/N của Eq. 7/10) ----
def ce_losses(model, prompts, targets, pad_id):
    """Eq. 8: ℓ(y*, p(.|x)) = −log p(str(y*) | x) (tổng log-prob các token của str(y*) và <eos>)."""
    lp, mask = target_logprobs(model, prompts, targets, pad_id)
    return -(lp * mask).sum(-1)


def raft_losses(model, scorer, prompts, labels, pad_id):
    """Eq. 10: (y* − Σ_{y∈Y_grid} p(str(y)|x)·y)²."""
    pred, _ = grid_predict(model, scorer, prompts, pad_id, RAFT_NORMALIZE)
    return (labels - pred) ** 2


def head_losses(model, head, prompts, labels, pad_id):
    """Table 2: (ŷ(x) − y*)² với ŷ = b + f_out(x)_{v*,L}."""
    return (head_predict(model, head, prompts, pad_id) - labels) ** 2

## Phần 8. Adafactor đúng thiết lập App. D
App. D: "*The parameters for Adafactor are: ε1 = 10⁻³⁰, ε2 = 10⁻³, decay rate = 0.8*", LR hằng. Đây đúng là tên và giá trị mặc định của `t5x/adafactor.py` (`epsilon1=1e-30, epsilon2=1e-3, decay_rate=0.8`, cùng `multiply_by_parameter_scale=True, clipping_threshold=1.0, beta1=None, min_dim_size_to_factor=128`). Ở T5X, ε2 *chỉ* có tác dụng khi `multiply_by_parameter_scale=True` (bước = lr·max(ε2, RMS(θ))) ⇒ notebook dùng thiết lập đó. Paper không nói rõ cờ này — đổi `ADAFACTOR_KW["multiply_by_parameter_scale"]=False` để dùng LR tuyệt đối.

Ánh xạ sang `transformers.optimization.Adafactor`: `lr=LR, eps=(1e-30, 1e-3), decay_rate=-0.8` (**dấu âm**: HF tính β̂₂ = 1 − t^{decay_rate}), `clip_threshold=1.0, beta1=None, weight_decay=0.0, scale_parameter=True, relative_step=False, warmup_init=False`. Notebook dùng bản port `AdafactorT5X` (để thêm *stochastic rounding* cho trọng số bf16); Phần 9 kiểm tra nó cho kết quả trùng với HF Adafactor.

In [ ]:
def stochastic_round_to_bf16(x: torch.Tensor) -> torch.Tensor:
    """Làm tròn ngẫu nhiên fp32 -> bf16 (kỳ vọng không lệch): cộng nhiễu đều vào 16 bit thấp rồi cắt."""
    xi = x.contiguous().view(torch.int32)
    noise = torch.randint(0, 1 << 16, xi.shape, device=x.device, dtype=torch.int32)
    return ((xi + noise) & -65536).view(torch.float32).to(torch.bfloat16)


class AdafactorT5X(torch.optim.Optimizer):
    """Adafactor (Shazeer & Stern, 2018), port PyTorch của t5x/adafactor.py với đúng tên/ý nghĩa siêu tham số
    mà App. D dùng: epsilon1=1e-30, epsilon2=1e-3, decay_rate=0.8, (mặc định T5X) clipping_threshold=1.0,
    multiply_by_parameter_scale=True, không momentum (beta1=None), không weight decay, LR hằng.
      β̂2_t = 1 − t^(−decay_rate);  V̂ ≈ factored(EMA[g² + ε1]);  U = g/√V̂;  U ← U / max(1, RMS(U)/d);
      θ ← θ − lr · max(ε2, RMS(θ)) · U      (hệ số max(ε2, RMS(θ)) chỉ khi multiply_by_parameter_scale)
    Tương đương transformers.optimization.Adafactor(lr=lr, eps=(1e-30, 1e-3), decay_rate=-0.8, clip_threshold=1.0,
    beta1=None, weight_decay=0.0, scale_parameter=True, relative_step=False, warmup_init=False) với ma trận có
    cả hai chiều >= 128 (được kiểm tra trong ô unit check)."""

    def __init__(self, params, lr, eps1=1e-30, eps2=1e-3, decay_rate=0.8, clipping_threshold=1.0,
                 multiply_by_parameter_scale=True, min_dim_size_to_factor=128, factored=True,
                 stochastic_rounding=False):
        super().__init__(params, dict(lr=lr, eps1=eps1, eps2=eps2, decay_rate=decay_rate,
                                      clipping_threshold=clipping_threshold,
                                      multiply_by_parameter_scale=multiply_by_parameter_scale,
                                      min_dim_size_to_factor=min_dim_size_to_factor, factored=factored,
                                      stochastic_rounding=stochastic_rounding))

    @staticmethod
    def _factored_dims(shape, group):
        if not group["factored"] or len(shape) != 2:
            return None
        d1, d0 = sorted(range(2), key=lambda d: shape[d])        # d0 = chiều lớn nhất, d1 = chiều lớn thứ hai
        return None if shape[d1] < group["min_dim_size_to_factor"] else (d1, d0)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad.float()
                st = self.state[p]
                if not st:
                    st["step"] = 0
                    st["fd"] = self._factored_dims(p.shape, group)
                    if st["fd"] is not None:
                        d1, d0 = st["fd"]
                        st["v_row"] = torch.zeros(p.shape[d1], device=p.device)   # trung bình theo d0
                        st["v_col"] = torch.zeros(p.shape[d0], device=p.device)   # trung bình theo d1
                    else:
                        st["v"] = torch.zeros_like(g)
                st["step"] += 1
                decay = 1.0 - st["step"] ** (-group["decay_rate"])
                mix = 1.0 - decay
                p32 = p.detach().float()
                scale = group["lr"]
                if group["multiply_by_parameter_scale"]:
                    scale = scale * p32.pow(2).mean().sqrt().clamp(min=group["eps2"])
                g2 = g * g + group["eps1"]
                if st["fd"] is not None:
                    d1, d0 = st["fd"]
                    st["v_row"].mul_(decay).add_(g2.mean(dim=d0), alpha=mix)
                    st["v_col"].mul_(decay).add_(g2.mean(dim=d1), alpha=mix)
                    row_factor = (st["v_row"] / st["v_row"].mean()).rsqrt()
                    y = g * row_factor.unsqueeze(d0) * st["v_col"].rsqrt().unsqueeze(d1)
                else:
                    st["v"].mul_(decay).add_(g2, alpha=mix)
                    y = g * st["v"].rsqrt()
                del g2
                if group["clipping_threshold"] is not None:
                    y = y / (y.pow(2).mean().sqrt() / group["clipping_threshold"]).clamp(min=1.0)
                new = p32 - scale * y
                if p.dtype == torch.bfloat16 and group["stochastic_rounding"]:
                    p.copy_(stochastic_round_to_bf16(new))
                else:
                    p.copy_(new)

## Phần 9. Unit check (chạy trên mô hình tí hon, CPU, vài giây)
1. Predictor RAFT một-chữ-số (đường nhanh) = $\sum_y y\cdot\mathrm{softmax}(\text{logits đầy đủ})[\mathrm{token}(y)]$ tính ngây thơ bằng forward HF.
2. Loss RAFT = Eq. 10 tính tay; ví dụ đồ chơi p = (0.1, 0.2, 0.3, 0.2, 0.1) trên '1'…'5' ⇒ ŷ = 2.7, với y* = 3 thì ℓ = 0.09 (chuẩn hoá: ŷ = 3.0).
3. Trie cho lưới 0.0..5.0 = chấm điểm từng ứng viên. 4. Loss CE (Eq. 8) = từ logits đầy đủ. 5. Predictive head = b + logit đầy đủ tại (v*, L).
6. `AdafactorT5X` = HF Adafactor (5 bước); stochastic rounding không lệch. 7. `float(z)`. 8. Dropout chỉ bật khi train; LoRA chạy được.
9. Đường autocast bf16 (như trên GPU): các loss, backward, generate chạy được; LM head và Σ y·p giữ fp32.

In [ ]:
def run_unit_checks():
    """Kiểm tra trên mô hình Gemma-2 tí hon (CPU-friendly, không cần mạng). Ném AssertionError nếu sai."""
    global DEVICE
    saved_device, DEVICE = DEVICE, torch.device("cpu")
    try:
        tk = build_char_tokenizer()
        m = build_tiny_gemma2(tk, seed=123).eval()
        pad = tk.pad_token_id
        texts = ["The similarity ... is equal to ", "short x is equal to "]   # độ dài khác nhau => có padding
        prompts = [tk(t)["input_ids"] for t in texts]
        grid = [1, 2, 3, 4, 5]
        scorer = GridScorer(tk, [str(v) for v in grid], grid)
        assert scorer.single_token and len(scorer.covers) == 1

        # (1) Predictor RAFT một-chữ-số == Σ_y y·softmax(f_out(x)_{:,L})[token(y)], tính "ngây thơ" bằng forward HF đầy đủ
        with torch.no_grad():
            fast, _ = grid_predict(m, scorer, prompts, pad)
            naive = []
            for p in prompts:
                logits = m(input_ids=torch.tensor([p])).logits[0, -1].float()     # HF đã áp soft-capping
                probs = torch.softmax(logits, -1)
                naive.append(sum(y * probs[tk.convert_tokens_to_ids(str(y))] for y in grid))
            naive = torch.stack(naive)
        torch.testing.assert_close(fast, naive, atol=1e-5, rtol=1e-5)

        # (2) Loss RAFT == Eq. 10 tính tay: (1/N) Σ (y* − Σ_y p(str(y)|x)·y)²
        labels = torch.tensor([3.4, 0.6])
        with torch.no_grad():
            got = raft_losses(m, scorer, prompts, labels, pad).mean()
        torch.testing.assert_close(got, ((labels - naive) ** 2).mean(), atol=1e-5, rtol=1e-5)
        # (2b) ví dụ đồ chơi với xác suất cho sẵn: p = (.1,.2,.3,.2,.1) trên '1'..'5' (tổng 0.9)
        #      ŷ = .1·1+.2·2+.3·3+.2·4+.1·5 = 2.7 ; y* = 3 => ℓ_RAFT = 0.09 ; chuẩn hoá: ŷ = 2.7/0.9 = 3.0
        probs = torch.tensor([[.1, .2, .3, .2, .1]])
        vals = torch.tensor([1., 2., 3., 4., 5.])
        yhat = probs.log().exp() @ vals
        torch.testing.assert_close(yhat, torch.tensor([2.7]))
        torch.testing.assert_close(((torch.tensor([3.0]) - yhat) ** 2).mean(), torch.tensor(0.09))
        torch.testing.assert_close(torch.softmax(probs.log(), -1) @ vals, torch.tensor([3.0]))

        # (3) Lưới nhiều token (0.0..5.0, có <eos>) qua trie == chấm điểm từng ứng viên bằng forward đầy đủ
        fine_vals = [round(0.1 * i, 1) for i in range(51)]
        fine = GridScorer(tk, [repr(v) for v in fine_vals], fine_vals, append_eos=True)
        assert len(fine.covers) == 51
        with torch.no_grad():
            lp_fast = fine.log_probs(m, prompts[:1], pad)[0]
            p0 = prompts[0]
            ref = []
            for c in fine.candidates:
                full = torch.log_softmax(m(input_ids=torch.tensor([p0 + list(c)])).logits[0].float(), -1)
                ref.append(sum(full[len(p0) - 1 + j, t] for j, t in enumerate(c)))
        torch.testing.assert_close(lp_fast, torch.stack(ref), atol=1e-4, rtol=1e-4)

        # (4) Loss CE (chỉ tính logits ở vị trí đích) == Eq. 8 từ logits đầy đủ
        targets = [tk("3.8", add_special_tokens=False)["input_ids"] + [tk.eos_token_id],
                   tk("0.25", add_special_tokens=False)["input_ids"] + [tk.eos_token_id]]
        with torch.no_grad():
            got = ce_losses(m, prompts, targets, pad)
            ref = []
            for p, t in zip(prompts, targets):
                full = torch.log_softmax(m(input_ids=torch.tensor([p + t])).logits[0].float(), -1)
                ref.append(-sum(full[len(p) - 1 + j, tt] for j, tt in enumerate(t)))
        torch.testing.assert_close(got, torch.stack(ref), atol=1e-4, rtol=1e-4)

        # (5) Predictive head == b + logits đầy đủ tại (v*, L)
        head = SpecialTokenHead(tk.convert_tokens_to_ids("<unused0>"), bias_init=0.7)
        with torch.no_grad():
            got = head_predict(m, head, prompts, pad)
            ref = torch.stack([0.7 + m(input_ids=torch.tensor([p])).logits[0, -1, head.token_id].float() for p in prompts])
        torch.testing.assert_close(got, ref, atol=1e-5, rtol=1e-5)

        # (6) Adafactor (port T5X) == transformers Adafactor(scale_parameter=True, relative_step=False, decay_rate=-0.8)
        from transformers.optimization import Adafactor as HFAdafactor
        torch.manual_seed(0)
        net_a = torch.nn.Sequential(torch.nn.Linear(128, 256), torch.nn.Tanh(), torch.nn.Linear(256, 128))
        net_b = copy.deepcopy(net_a)
        opt_a = AdafactorT5X(net_a.parameters(), lr=1e-2, **ADAFACTOR_KW)
        opt_b = HFAdafactor(net_b.parameters(), lr=1e-2, eps=(1e-30, 1e-3), clip_threshold=1.0, decay_rate=-0.8,
                            beta1=None, weight_decay=0.0, scale_parameter=True, relative_step=False, warmup_init=False)
        for s in range(5):
            xb = torch.randn(32, 128, generator=torch.Generator().manual_seed(s))
            for net, opt in ((net_a, opt_a), (net_b, opt_b)):
                opt.zero_grad()
                net(xb).pow(2).mean().backward()
                opt.step()
        for pa, pb in zip(net_a.parameters(), net_b.parameters()):
            torch.testing.assert_close(pa, pb, atol=1e-6, rtol=1e-5)
        # làm tròn ngẫu nhiên không lệch: E[SR(x)] ≈ x
        x = torch.full((200000,), 1.0 + 2 ** -10)
        assert abs(stochastic_round_to_bf16(x).float().mean().item() - x[0].item()) < 2e-4

        # (7) float(z) — §2.2
        assert float_of(" 3.8") == (3.8, True) and float_of("4.25 because") == (4.25, True)
        assert float_of("banana") == (NON_NUMERIC_VALUE, False) and float_of("") == (NON_NUMERIC_VALUE, False)
        assert float_of("45")[1] is (not OUT_OF_RANGE_TO_DEFAULT)

        # (8) Dropout residual chỉ hoạt động khi train; LoRA chạy được (fallback, DEVIATION)
        m2 = build_tiny_gemma2(tk, seed=1)
        handles = add_dropout(m2, 0.5, "residual")
        ids = torch.tensor([prompts[0]])
        m2.eval()
        with torch.no_grad():
            e1, e2 = m2(input_ids=ids).logits, m2(input_ids=ids).logits
        m2.train()
        with torch.no_grad():
            t1 = m2(input_ids=ids).logits
        assert torch.equal(e1, e2) and not torch.allclose(e1, t1)
        for h_ in handles:
            h_.remove()
        from peft import LoraConfig, get_peft_model
        pm = get_peft_model(build_tiny_gemma2(tk, seed=2), LoraConfig(r=4, lora_alpha=8, target_modules="all-linear",
                                                                       task_type="CAUSAL_LM"))
        hd = SpecialTokenHead(head.token_id)
        loss = raft_losses(pm, scorer, prompts, labels, pad).mean() + head_losses(pm, hd, prompts, labels, pad).mean()
        loss.backward()
        lora_grads = [p.grad for n, p in pm.named_parameters() if "lora_" in n and p.requires_grad]
        assert lora_grads and all(g_ is not None for g_ in lora_grads) and hd.bias.grad is not None

        # (9) Đường autocast bf16 (như trên GPU): loss + backward + generate chạy được, dtype đầu ra fp32,
        #     và gần kết quả fp32 (LM head và Σ y·p được tính ở fp32)
        m3 = build_tiny_gemma2(tk, seed=3)
        with torch.no_grad():
            ref_r = raft_losses(m3, scorer, prompts, labels, pad)
            ref_h = head_predict(m3, head, prompts, pad)
        with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
            l_r = raft_losses(m3, scorer, prompts, labels, pad)
            l_c = ce_losses(m3, prompts, targets, pad)
            l_h = head_losses(m3, head, prompts, labels, pad)
            yh = head_predict(m3, head, prompts, pad)
            gen = m3.generate(input_ids=torch.tensor([prompts[0]]), max_new_tokens=3, do_sample=False,
                              pad_token_id=pad, eos_token_id=tk.eos_token_id)
        assert l_r.dtype == l_c.dtype == l_h.dtype == yh.dtype == torch.float32 and gen.shape[1] == len(prompts[0]) + 3
        (l_r.sum() + l_c.sum() + l_h.sum()).backward()
        assert all(torch.isfinite(p.grad).all() for p in m3.parameters() if p.grad is not None)
        torch.testing.assert_close(l_r.detach(), ref_r, rtol=5e-2, atol=5e-2)
        torch.testing.assert_close(yh.detach(), ref_h, rtol=5e-2, atol=5e-2)
    finally:
        DEVICE = saved_device
    print("ALL UNIT CHECKS PASSED")


run_unit_checks()

## Phần 10. Tokenizer thật + kiểm tra input x
In 4 token cuối của x, token của từng y ∈ 𝒴_grid (phải là **1 token** để dùng công thức một-chữ-số của §4.2), token v*, và kiểm tra tokens(x + str(y)) = tokens(x) + tokens(str(y)). Với Gemma-2 và `ANSWER_PREFIX=" "` kỳ vọng: `[..., '▁equal', '▁to', '▁']`, mỗi chữ số `'1'..'5'` một token.

In [ ]:
TOK = load_tokenizer()
PAD_ID = TOK.pad_token_id
HEAD_TOKEN_ID = TOK.convert_tokens_to_ids(HEAD_TOKEN)
assert HEAD_TOKEN_ID is not None and HEAD_TOKEN_ID != TOK.unk_token_id, f"Không tìm thấy token {HEAD_TOKEN}"

RAFT_SCORER = GridScorer(TOK, [str(v) for v in RAFT_GRID], RAFT_GRID, append_eos=False)
FINE_GRID = [round(0.1 * i, 1) for i in range(51)]           # chỉ cho cột phụ RAIL-lưới của mô hình CE
FINE_SCORER = GridScorer(TOK, [format_target(v) for v in FINE_GRID], FINE_GRID, append_eos=True)


def show_tokens(ids):
    return [TOK.convert_ids_to_tokens(i) for i in ids]


_ex = VAL[0]
_x = encode_prompt(TOK, _ex)
print("4 token cuối của x:", show_tokens(_x[-4:]))
print("Token của lưới RAFT:", {s: show_tokens(c) for s, c in zip(RAFT_SCORER.strings, RAFT_SCORER.candidates)})
print("RAFT một-chữ-số (mỗi y là đúng 1 token):", RAFT_SCORER.single_token)
print(f"Token v* của predictive head: {HEAD_TOKEN} -> id {HEAD_TOKEN_ID}")
_s = format_target(3.8)
_joint = (TOK(_ex.prompt() + ANSWER_PREFIX + _s)["input_ids"] if not USE_CHAT_TEMPLATE else None)
if _joint is not None:
    _ok = _joint == _x + TOK(_s, add_special_tokens=False)["input_ids"]
    print(f"tokens(x + '{_s}') == tokens(x) + tokens('{_s}'):", _ok)
    if not _ok:
        print("  CẢNH BÁO: tokenization không ghép được; p(str(y)|x) sẽ khác cách mô hình nhìn thấy văn bản liền.")
print("Ví dụ đích CE:", show_tokens(encode_target(TOK, 3.8)))

## Phần 11. Đánh giá và chọn mô hình
* Chỉ số: **RMSE** (§5.1). Pearson/Spearman chỉ in thêm (Table 12).
* Mỗi `EVAL_EVERY` bước (và ở bước 0) đánh giá trên **val (863)** và **test (1,500)**; với mỗi decoder, chọn bước có val RMSE nhỏ nhất và báo cáo test RMSE tại bước đó (App. D: "*select the best step using the held out validation set*"). Ghi test ở mọi lần eval để không phải lưu checkpoint — việc chọn chỉ dùng val.
* Với mô hình CE, hai cột (greedy, RAIL) đến từ **cùng** các run; bước tốt nhất chọn **riêng** cho từng decoder (paper không nói).
* Cột phụ: `rail_grid_digits` (Eq. 5 với 𝒴_grid zero-shot — đúng bằng RAFT ở bước 0, §E.5 p.26) và `rail_grid_fine` (Eq. 5 với lưới 0.0..5.0 + `<eos>`, đắt, tắt mặc định).

In [ ]:
def metrics(preds, labels):
    p, y = np.asarray(preds, dtype=np.float64), np.asarray(labels, dtype=np.float64)
    out = {"rmse": float(np.sqrt(np.mean((p - y) ** 2)))}            # chỉ số chính của paper (§5.1)
    ok = np.std(p) > 0
    out["pearson"] = float(pearsonr(p, y)[0]) if ok else float("nan")   # phụ (Table 12)
    out["spearman"] = float(spearmanr(p, y)[0]) if ok else float("nan")
    return out


# Các "decoder" (predictor) được đánh giá cho từng phương pháp. Tên cột Table 3 ở §14.
DECODERS = {
    "zero_shot": ["greedy", "rail_sampling", "rail_grid_digits"] + (["rail_grid_fine"] if CE_EVAL_RAIL_GRID else []),
    "ce": ["greedy", "rail_sampling"] + (["rail_grid_fine"] if CE_EVAL_RAIL_GRID else []),
    "head": ["head"],
    "raft": ["raft"],
}


@torch.no_grad()
def evaluate(method, model, examples, head=None, seed=EVAL_SAMPLING_SEED):
    """Dự đoán + chỉ số cho mọi decoder của `method` trên `examples`. Dropout tắt (model.eval())."""
    was_training = model.training
    model.eval()
    prompts = [encode_prompt(TOK, e) for e in examples]
    labels = [e.label for e in examples]
    preds, extra = {}, {}
    devices = [torch.cuda.current_device()] if DEVICE.type == "cuda" else []
    with torch.random.fork_rng(devices=devices):       # không làm xáo trộn RNG của vòng huấn luyện
        torch.manual_seed(seed)
        for dec in DECODERS[method]:
            if dec == "greedy":
                preds[dec], extra[dec] = greedy_predict(model, TOK, prompts)
            elif dec == "rail_sampling":
                preds[dec], extra[dec] = rail_sampling_predict(model, TOK, prompts)
            else:
                out, mass = [], []
                bs = 4 if dec == "rail_grid_fine" else EVAL_BATCH_SIZE
                for i in range(0, len(prompts), bs):
                    chunk = prompts[i:i + bs]
                    with autocast_ctx():
                        if dec == "head":
                            out.append(head_predict(model, head, chunk, PAD_ID).float())
                        else:
                            scorer = FINE_SCORER if dec == "rail_grid_fine" else RAFT_SCORER
                            normalize = RAFT_NORMALIZE and dec in ("raft", "rail_grid_digits")
                            yhat, logp = grid_predict(model, scorer, chunk, PAD_ID, normalize)
                            out.append(yhat.float())
                            mass.append(logp.float().exp().sum(-1))
                preds[dec] = torch.cat(out).cpu().tolist()
                extra[dec] = float(torch.cat(mass).mean()) if mass else None
    if was_training:
        model.train()
    res = {}
    for dec, p in preds.items():
        res[dec] = metrics(p, labels)
        if dec in ("greedy", "rail_sampling"):
            res[dec]["unparsed_rate"] = extra[dec]      # tỉ lệ đầu ra bị float(z)=0.0 (§2.2) — rất nhạy với RMSE
        elif extra[dec] is not None:
            res[dec]["grid_mass"] = extra[dec]          # Σ_{y∈Y_grid} p(str(y)|x) trung bình
    return res, preds

## Phần 12. Cột 1–2: Zero-shot (standard decoding = greedy; RAIL = 32 mẫu, T = 1)
Không huấn luyện ⇒ một giá trị (không có ± như paper). Trước đó in **khối xác suất trên chữ số ở bước 0** cho x có và không có hậu tố `' '` — với cấu hình đúng, khối này phải lớn (Fig. 7).

In [ ]:
def save_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=1)
    os.replace(tmp, path)


@torch.no_grad()
def digit_mass_check(model, examples, n=64):
    """Kiểm tra: ở bước 0, khối xác suất trên các token chữ số '0'..'9' tại vị trí L (ngay sau x).
    So sánh x = prompt + ANSWER_PREFIX với prompt không có hậu tố. Fig. 7 (p.31) cho thấy trước fine-tune gần như
    toàn bộ khối xác suất nằm trên chữ số => thiết lập đúng phải cho khối lớn."""
    model.eval()
    digits = GridScorer(TOK, [str(d) for d in range(10)], list(range(10)))
    exs = examples[:n]
    variants = {repr(ANSWER_PREFIX): [encode_prompt(TOK, e) for e in exs]}
    if ANSWER_PREFIX and not USE_CHAT_TEMPLATE:
        variants["'' (không hậu tố)"] = [TOK(e.prompt())["input_ids"] for e in exs]
    report = {}
    for name, prompts in variants.items():
        lps = []
        for i in range(0, len(prompts), EVAL_BATCH_SIZE):
            with autocast_ctx():
                lps.append(digits.log_probs(model, prompts[i:i + EVAL_BATCH_SIZE], PAD_ID).float())
        p = torch.cat(lps).exp()
        grid_idx = [d for d in RAFT_GRID if 0 <= d <= 9]
        report[name] = {"digit_mass": float(p.sum(-1).mean()), "grid_mass": float(p[:, grid_idx].sum(-1).mean()),
                        "mean_digit_dist": p.mean(0).cpu().tolist()}
        print(f"hậu tố {name:18s}: Σ p(chữ số 0-9) = {p.sum(-1).mean():.3f} | Σ p(Y_grid) = {p[:, grid_idx].sum(-1).mean():.3f}"
              f" | phân phối TB: {np.round(p.mean(0).cpu().numpy(), 3).tolist()}")
    if report[repr(ANSWER_PREFIX)]["digit_mass"] < 0.5:
        print("CẢNH BÁO: khối xác suất trên chữ số ở bước 0 nhỏ — kiểm tra ANSWER_PREFIX / USE_CHAT_TEMPLATE."
              + (" (Bình thường với mô hình ngẫu nhiên của SMOKE_TEST.)" if SMOKE_TEST else ""))
    return report


ZS_PATH = os.path.join(OUTPUT_DIR, "zero_shot.json")
if "zero_shot" in METHODS:
    if SKIP_EXISTING and os.path.exists(ZS_PATH):
        ZERO_SHOT = json.load(open(ZS_PATH))
        print("Đã có", ZS_PATH)
    else:
        t0 = time.time()
        model = load_model()
        mass_report = digit_mass_check(model, VAL)
        val_res, _ = evaluate("zero_shot", model, VAL)
        test_res, test_preds = evaluate("zero_shot", model, TEST)
        ZERO_SHOT = {"model": MODEL_NAME, "method": "zero_shot", "val": val_res, "test": test_res,
                     "test_predictions": test_preds, "digit_mass_check": mass_report, "seconds": time.time() - t0}
        save_json(ZERO_SHOT, ZS_PATH)
        del model
        free_memory()
    for dec, r in ZERO_SHOT["test"].items():
        print(f"zero-shot {dec:18s} test RMSE = {r['rmse']:.4f}  " + json.dumps({k: v for k, v in r.items() if k != 'rmse'}))

## Phần 13. Cột 3–6: vòng huấn luyện chung cho `ce`, `head`, `raft`
Mỗi run = (phương pháp, LR, seed). Seed điều khiển thứ tự dữ liệu và mặt nạ dropout (tập train/val cố định theo `SPLIT_SEED`). Batch 16 (tích luỹ gradient nếu `MICRO_BATCH_SIZE < 16`); loss = tổng loss từng ví dụ / 16 = trung bình 1/N của Eq. 7 / Eq. 10. Không có gradient clipping toàn cục (T5X mặc định không có; Adafactor tự *update clipping* với ngưỡng 1.0). Mỗi run lưu JSON: lịch sử val/test, bước tốt nhất từng decoder, dự đoán test tại bước đó, loss huấn luyện, thời gian/bước, cấu hình.

In [ ]:
@dataclass
class TrainItem:
    prompt_ids: List[int]
    target_ids: List[int]
    label: float


def run_path(method, lr, seed):
    return os.path.join(RUNS_DIR, f"{method}_lr{lr:g}_seed{seed}.json")


def batch_stream(n, seed):
    """Xáo trộn theo epoch (random.Random(seed)); mỗi batch BATCH_SIZE ví dụ, bỏ phần dư cuối epoch."""
    rng = random.Random(seed)
    while True:
        order = list(range(n))
        rng.shuffle(order)
        for i in range(0, n - BATCH_SIZE + 1, BATCH_SIZE):
            yield order[i:i + BATCH_SIZE]


def train_one_run(method, lr, seed):
    """Fine-tune một lần (method ∈ {ce, head, raft}), eval val+test mỗi EVAL_EVERY bước (cả bước 0),
    chọn bước tốt nhất theo val RMSE cho từng decoder (App. D), lưu JSON."""
    path = run_path(method, lr, seed)
    if SKIP_EXISTING and os.path.exists(path):
        print("Bỏ qua (đã có):", path)
        return json.load(open(path))
    set_seed(seed)
    model = load_model()
    head = SpecialTokenHead(HEAD_TOKEN_ID, HEAD_BIAS_INIT).to(DEVICE) if method == "head" else None
    params = [p for p in model.parameters() if p.requires_grad] + (list(head.parameters()) if head is not None else [])
    opt = AdafactorT5X(params, lr=lr, stochastic_rounding=STOCHASTIC_ROUNDING, **ADAFACTOR_KW)
    n_trainable = sum(p.numel() for p in params)
    data = [TrainItem(encode_prompt(TOK, e), encode_target(TOK, e.label), e.label) for e in TRAIN]
    stream = batch_stream(len(data), seed)
    history, best, loss_log = [], {}, []
    evals_since_best, t_start, t_train = 0, time.time(), 0.0

    def do_eval(step):
        nonlocal evals_since_best
        val_res, _ = evaluate(method, model, VAL, head)
        test_res, test_preds = evaluate(method, model, TEST, head)
        history.append({"step": step, "val": val_res, "test": test_res})
        improved = False
        for dec in DECODERS[method]:
            if dec not in best or val_res[dec]["rmse"] < best[dec]["val_rmse"]:
                best[dec] = {"step": step, "val_rmse": val_res[dec]["rmse"], "test_rmse": test_res[dec]["rmse"],
                             "test": test_res[dec], "val": val_res[dec], "test_predictions": test_preds[dec]}
                improved = True
        evals_since_best = 0 if improved else evals_since_best + 1
        print(f"  [{method} lr={lr:g} seed={seed}] step {step:6d} | " + " | ".join(
            f"{d}: val {val_res[d]['rmse']:.4f} test {test_res[d]['rmse']:.4f}" for d in DECODERS[method]), flush=True)

    model.train()
    do_eval(0)                              # bước 0: RAFT@0 == RAIL lưới zero-shot (§E.5 p.26)
    for step in range(1, MAX_STEPS + 1):
        t0 = time.time()
        batch = [data[i] for i in next(stream)]
        opt.zero_grad(set_to_none=True)
        total = 0.0
        for j in range(0, BATCH_SIZE, MICRO_BATCH_SIZE):
            mb = batch[j:j + MICRO_BATCH_SIZE]
            prompts = [it.prompt_ids for it in mb]
            labels = torch.tensor([it.label for it in mb], device=DEVICE, dtype=torch.float32)
            with autocast_ctx():
                if method == "ce":
                    losses = ce_losses(model, prompts, [it.target_ids for it in mb], PAD_ID)
                elif method == "raft":
                    losses = raft_losses(model, RAFT_SCORER, prompts, labels, PAD_ID)
                else:
                    losses = head_losses(model, head, prompts, labels, PAD_ID)
            loss = losses.float().sum() / BATCH_SIZE          # trung bình trên batch 16 (Eq. 7 / Eq. 10)
            loss.backward()
            total += loss.item()
        opt.step()
        t_train += time.time() - t0
        if not math.isfinite(total):
            print("  Loss không hữu hạn — dừng run này.")
            break
        if step % 10 == 0 or step == 1:
            loss_log.append([step, total])
        if step % EVAL_EVERY == 0 or step == MAX_STEPS:
            do_eval(step)
            if EARLY_STOP_PATIENCE is not None and evals_since_best >= EARLY_STOP_PATIENCE:
                print(f"  Early stop ở bước {step} (DEVIATION: paper chạy 200K bước).")
                break
    result = {"model": MODEL_NAME, "method": method, "lr": lr, "seed": seed, "max_steps": MAX_STEPS,
              "eval_every": EVAL_EVERY, "trainable_params": n_trainable,
              "sec_per_step": t_train / max(1, step), "seconds_total": time.time() - t_start,
              "best": best, "history": history, "train_loss": loss_log,
              "config": {k: globals()[k] for k in CONFIG_KEYS}}
    save_json(result, path)
    del model, head, opt
    free_memory()
    return result


CONFIG_KEYS = ["MODEL_NAME", "USE_CHAT_TEMPLATE", "BATCH_SIZE", "MICRO_BATCH_SIZE", "MAX_STEPS", "EVAL_EVERY",
               "EARLY_STOP_PATIENCE", "DROPOUT", "DROPOUT_MODE", "ADAFACTOR_KW", "PARAM_DTYPE", "STOCHASTIC_ROUNDING",
               "COMPUTE_DTYPE", "GRADIENT_CHECKPOINTING", "ATTN_IMPLEMENTATION", "USE_LORA", "LORA_R", "LORA_ALPHA",
               "DATA_SOURCE", "SPLIT_SEED", "N_VAL", "N_TRAIN", "TARGET_DECIMALS", "ANSWER_PREFIX", "RAFT_GRID",
               "RAFT_NORMALIZE", "HEAD_TOKEN", "HEAD_BIAS_INIT", "RAIL_NUM_SAMPLES", "RAIL_TEMPERATURE",
               "MAX_NEW_TOKENS", "NON_NUMERIC_VALUE", "OUT_OF_RANGE_TO_DEFAULT", "PARSE_MODE", "EVAL_SAMPLING_SEED"]

## Phần 14. Chạy tất cả
**Ước lượng chi phí (thô, chưa đo trên GPU — notebook in `s/bước` thật sau mỗi run):** Gemma-2 2B (~2.6e9 tham số), batch 16 × ~78 token (x trung bình 73.9 token, tối đa 150, đo bằng tokenizer Gemma-2 trên STS-B) ≈ 6·2.6e9·1,250 ≈ 2e13 FLOP/bước ⇒ cỡ 0.2–0.5 s/bước trên A100 (bf16), cộng chi phí Adafactor trên ~2.6e9 tham số fp32. Theo paper: 200K bước ≈ 11–28 giờ/run; 3 phương pháp × 3 LR × 3 seed = 27 run ⇒ cỡ 300–750 GPU-giờ cho 2B, gấp ~3–4 lần cho 9B — không thực tế với một GPU.

**Mặc định thực dụng `MAX_STEPS = 6000`, eval mỗi 200 bước (DEVIATION):** Fig. 4 (p.29) cho thấy trên STSB với 100% dữ liệu, số bước tới hội tụ (theo val) là ~3,500 cho RAFT và ~4,200 cho predictive head (thanh sai số tới ~4,500); Fig. 5 (p.30) cho thấy RMSE gần như bão hoà sau ~1k bước trên Wireless. Paper không có đường hội tụ cho CE ⇒ kiểm tra `history` và tăng `MAX_STEPS` nếu val RMSE của CE vẫn đang giảm ở cuối. Ước tính 2B: ~20–50 phút/run (RAFT/head) + thời gian eval (31 lần × val+test); CE đắt hơn vì mỗi lần eval sinh 32 mẫu × 2,363 ví dụ. Muốn rẻ hơn nữa: chọn LR bằng một seed (`SEEDS=[0]`), rồi chạy seed 1, 2 với LR đã chọn (`LEARNING_RATES=[lr*]`) — bảng tổng hợp vẫn chọn LR theo val trung bình trên các seed có sẵn.

In [ ]:
ALL_RUNS = []
for method in [m for m in ("raft", "head", "ce") if m in METHODS]:
    for lr in LEARNING_RATES:
        for seed in SEEDS:
            print(f"=== {method} | lr={lr:g} | seed={seed} ===")
            r = train_one_run(method, lr, seed)
            ALL_RUNS.append(r)
            print(f"    xong: {r['sec_per_step']:.3f} s/bước; best: " +
                  ", ".join(f"{d}@{b['step']} val {b['val_rmse']:.4f} test {b['test_rmse']:.4f}" for d, b in r["best"].items()))

## Phần 15. Tổng hợp theo bố cục Table 3
Với mỗi cột: chọn LR có **val RMSE trung bình qua seed** nhỏ nhất (App. D), báo cáo **mean ± std** (ddof = `STD_DDOF`) của test RMSE tại bước tốt nhất của từng seed. Paper không nói LR được chọn chung hay theo từng seed và ddof — đây là lựa chọn của notebook. Bảng được lưu ở `OUTPUT_DIR/table3_stsb.md`, chi tiết ở `table3_stsb_summary.json`.

In [ ]:
TABLE3_COLUMNS = [   # (tên cột Table 3, method, decoder)
    ("Zero-shot standard decoding", "zero_shot", "greedy"),
    ("Zero-shot RAIL", "zero_shot", "rail_sampling"),
    ("Standard fine-tuning standard decoding", "ce", "greedy"),
    ("Standard fine-tuning RAIL", "ce", "rail_sampling"),
    ("Predictive head", "head", "head"),
    ("RAFT", "raft", "raft"),
]
EXTRA_COLUMNS = [
    ("Zero-shot RAIL lưới Y_grid (Eq. 5) = RAFT@bước 0", "zero_shot", "rail_grid_digits"),
    ("Zero-shot RAIL lưới 0.0..5.0 (Eq. 5)", "zero_shot", "rail_grid_fine"),
    ("Std FT + RAIL lưới 0.0..5.0 (Eq. 5)", "ce", "rail_grid_fine"),
]
PAPER_TABLE3_STSB = {   # Table 3 p.8 (RMSE; fine-tuned: mean ± std qua 3 lần huấn luyện)
    "gemma-2-2b": ["1.10", "1.05", "0.59 ± 0.01", "0.57 ± 0.00", "0.54 ± 0.00", "0.54 ± 0.01"],
    "gemma-2-9b": ["1.37", "1.29", "0.58 ± 0.00", "0.53 ± 0.00", "0.52 ± 0.00", "0.51 ± 0.01"],
}


def load_runs():
    runs = []
    for p in sorted(glob.glob(os.path.join(RUNS_DIR, "*.json"))):
        r = json.load(open(p))
        if r.get("model") == MODEL_NAME:
            runs.append(r)
    return runs


def summarize_column(runs, method, dec):
    """Chọn LR theo val RMSE trung bình qua seed (App. D: chọn LR trên validation), rồi báo cáo
    mean ± std của test RMSE (tại bước tốt nhất theo val của từng seed) với LR đó."""
    by_lr = {}
    for r in runs:
        if r["method"] == method and dec in r["best"]:
            by_lr.setdefault(r["lr"], []).append(r["best"][dec])
    if not by_lr:
        return None
    per_lr = {lr: {"val_mean": float(np.mean([b["val_rmse"] for b in bs])),
                   "test": [b["test_rmse"] for b in bs], "steps": [b["step"] for b in bs]} for lr, bs in by_lr.items()}
    lr_star = min(per_lr, key=lambda lr: per_lr[lr]["val_mean"])
    t = np.array(per_lr[lr_star]["test"])
    std = float(t.std(ddof=STD_DDOF)) if len(t) > STD_DDOF else float("nan")
    return {"lr": lr_star, "mean": float(t.mean()), "std": std, "n_seeds": len(t),
            "best_steps": per_lr[lr_star]["steps"], "per_lr": per_lr}


def build_table3():
    runs = load_runs()
    zs = json.load(open(ZS_PATH)) if os.path.exists(ZS_PATH) else None
    summary, cells = {}, []
    for name, method, dec in TABLE3_COLUMNS + EXTRA_COLUMNS:
        if method == "zero_shot":
            v = zs["test"].get(dec) if zs else None
            summary[name] = {"rmse": v["rmse"]} if v else None
            cells.append(f"{v['rmse']:.2f}" if v else "—")
        else:
            s = summarize_column(runs, method, dec)
            summary[name] = s
            cells.append("—" if s is None else (f"{s['mean']:.2f} ± {s['std']:.2f}" if s["n_seeds"] > 1
                                                else f"{s['mean']:.2f} (1 seed)"))
    key = next((k for k in PAPER_TABLE3_STSB if k in MODEL_NAME.lower()), None)
    names = [c[0] for c in TABLE3_COLUMNS]
    lines = ["| Hàng | " + " | ".join(names) + " |", "|---" * (len(names) + 1) + "|",
             f"| Chạy lại ({os.path.basename(MODEL_NAME)}) | " + " | ".join(cells[:len(names)]) + " |"]
    if key and not SMOKE_TEST:
        lines.append(f"| Paper Table 3 (Gemma-2 {key[-2:].upper()}) | " + " | ".join(PAPER_TABLE3_STSB[key]) + " |")
    table = "\n".join(lines)
    extra = "\n".join(f"- {n}: {c}" for (n, _, _), c in zip(EXTRA_COLUMNS, cells[len(names):]) if c != "—")
    detail = []
    for name, method, dec in TABLE3_COLUMNS + EXTRA_COLUMNS:
        s = summary.get(name)
        if s and "per_lr" in s:
            detail.append(f"- {name}: LR chọn = {s['lr']:g}; bước tốt nhất theo seed = {s['best_steps']}; " + "; ".join(
                f"lr {lr:g}: val TB {d['val_mean']:.4f}, test {np.round(d['test'], 4).tolist()}" for lr, d in sorted(s["per_lr"].items())))
    md = (f"## STSB — Table 3 (RMSE, test = GLUE validation 1,500)\n\n{table}\n\n"
          + (f"Cột phụ (không có trong Table 3):\n{extra}\n\n" if extra else "")
          + ("Chi tiết chọn LR:\n" + "\n".join(detail) + "\n" if detail else ""))
    if SMOKE_TEST:
        md = "**SMOKE_TEST: mô hình tí hon ngẫu nhiên + dữ liệu tổng hợp — số liệu KHÔNG có ý nghĩa so với paper.**\n\n" + md
    with open(os.path.join(OUTPUT_DIR, "table3_stsb.md"), "w") as f:
        f.write(md)
    save_json(summary, os.path.join(OUTPUT_DIR, "table3_stsb_summary.json"))
    return md


TABLE3_MD = build_table3()
print(TABLE3_MD)

## Phần 16. Ghi chú: chỗ paper không nói rõ và lựa chọn của notebook
| Chi tiết | Paper | Notebook |
|---|---|---|
| Checkpoint | "instruction-tuned" (§5.1) | `-it`; base là tuỳ chọn |
| Chat template | không nói | tắt (`USE_CHAT_TEMPLATE`) |
| Ranh giới x / số (dấu cách) | không nói | x = prompt + `' '` ⇒ chữ số là token kế tiếp |
| Độ chính xác str(y) cho CE | không nói | 2 chữ số thập phân |
| float(z) | 0.0 nếu z ∉ 𝒴 (§2.2) | số đầu tiên dòng đầu; ngoài [0,5] ⇒ 0.0; log tỉ lệ |
| "Std FT + RAIL" | không nói (32 mẫu hay chấm lưới) | 32 mẫu, T = 1 (như zero-shot RAIL); lưới 0.0..5.0 là cột phụ |
| 𝒴_grid STSB | {1..5} (§5.1); Fig. 7 vẽ 0–5 | {1..5}; {0..5} cho cùng kết quả khi không chuẩn hoá |
| v* của predictive head; khởi tạo b | không nói | `<unused0>`; b = 0 |
| Vị trí dropout | "dropout rate 0.1" | residual dropout |
| Cờ parameter-scale của Adafactor | không nói (ε2 được nêu) | bật (mặc định T5X) |
| Cách tách 4,887/863 | không nói (tổng 5,750 ≠ 5,749) | hoán vị seed 0; 4,886/863 |
| Độ chính xác số của trọng số | không nói | fp32 master + bf16 autocast (9B: bf16 + stochastic rounding) |
| Số bước | 200K | 6,000 (DEVIATION, Fig. 4) |
| Chọn LR, ddof của std | "chọn trên validation" | val trung bình qua seed; ddof = 1 |

Mâu thuẫn nhỏ trong paper (ghi để tránh so sánh nhầm): Table 5 (p.22) RAFT STSB Gemma-2 9B = 0.40 nhưng Table 3/12 = 0.51; Table 8 (STSB-1K) zero-shot 9B = 1.31 nhưng Table 3 = 1.37 (zero-shot không phụ thuộc tập train, cùng tập test 1,500); 4,887 + 863 = 5,750 so với 5,749 cặp train của GLUE.